# 02 - Text preprocessing

**Goal of this notebook.** A neural network reads numbers, not text. Here I turn the questions and the queries into
sequences of integers, and I measure what this conversion costs, because it sets the limits of what any model can reach.

```
text  ->  tokens  ->  ids  ->  padded batches  ->  network
```



Files written by this notebook:

| File | Role |
|---|---|
| `src/t2s/data/text.py` | normalization and tokenization |
| `src/t2s/data/format.py` | the format of the input and output sequences (query to tokens and back) |
| `src/t2s/data/vocab.py` | the vocabulary (token to id) |
| `src/t2s/data/dataset.py` | ids, truncation, padding and batches |

Tools from notebook 01 that are used here:

| Tool | What it does | Notebook 01 |
|---|---|---|
| `download_wikisql` | downloads and checks the data | section 1 |
| `load_split` / `Example` | loads a split; one question with its table and query | section 4 |
| `Query` | a query (column, aggregator, conditions) | section 3 |
| `SqliteExecutor` | runs a query on the database | section 6 |

## Setup


In [1]:
import json, os, sys
from pathlib import Path

NOTEBOOK_NUMBER = 2
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/text2sql")
    CODE = "/content/code"                                      # where the notebooks write their code
    DATA_ROOT = "/content/data"                                 # the dataset (downloaded in each session)
    CKPT_DIR = "/content/drive/MyDrive/text2sql_checkpoints"    # the saved models
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    CODE, DATA_ROOT, CKPT_DIR = str(PROJECT / "code"), str(PROJECT / "data"), str(PROJECT / "checkpoints")
os.chdir(PROJECT)   # the results are saved in the folder "results" of the project
os.makedirs(CKPT_DIR, exist_ok=True)


def rebuild_code(before):
    """Write again the code files of the notebooks 01 ... before-1 (their writefile cells)."""
    for path in sorted((PROJECT / "notebooks").glob("[0-9][0-9]_*.ipynb")):
        if int(path.name[:2]) >= before:
            continue
        for cell in json.loads(path.read_text())["cells"]:
            source = "".join(cell["source"])
            if cell["cell_type"] != "code" or not source.startswith("%%writefile"):
                continue
            first_line, _, body = source.partition("\n")
            words = first_line.split()
            target = Path(words[-1].replace("{CODE}", CODE))
            target.parent.mkdir(parents=True, exist_ok=True)
            with open(target, "a" if words[1] == "-a" else "w") as f:
                f.write(body if body.endswith("\n") else body + "\n")


for folder in ["src/t2s/data", "src/t2s/models", "src/t2s/decoding", "src/t2s/eval", "scripts", "configs"]:
    os.makedirs(f"{CODE}/{folder}", exist_ok=True)
rebuild_code(NOTEBOOK_NUMBER)
for package in ["src/t2s", "src/t2s/data", "src/t2s/models", "src/t2s/decoding", "src/t2s/eval"]:
    Path(f"{CODE}/{package}/__init__.py").touch()
sys.path.insert(0, f"{CODE}/src")

Mounted at /content/drive


In [2]:
import json
from pathlib import Path
from t2s.data.download import download_wikisql
from t2s.data.wikisql import load_split

data_dir = download_wikisql(DATA_ROOT)
splits = {name: load_split(data_dir, name) for name in ["train", "dev", "test"]}
print({name: len(examples) for name, examples in splits.items()})

out_dir = Path("results/phase2_text")
out_dir.mkdir(parents=True, exist_ok=True)
stats = {}

{'train': 56355, 'dev': 8421, 'test': 15878}


## 1. Normalization and tokenization

First the text is cut into small pieces, the *tokens*. Cutting only at spaces is not enough here, because punctuation
matters: `(ks)`, `w-l` and `2–0` appear in values and column names.

**The idea.** One regular expression, `TOKEN_RE`, cuts the text into three kinds of token: a **number** (kept whole when
it has a decimal point or a thousands comma, such as `3.5` or `1,200`), a **word** (letters, digits, underscore), or **any
other single character** (a punctuation mark or a symbol). So `Butler CC (KS)` becomes `butler`, `cc`, `(`, `ks`, `)`.

Each token also remembers **where it came from** in the original text. I need this in section 2 to give back the exact
characters of a value.

In [3]:
%%writefile {CODE}/src/t2s/data/text.py
"""Normalization and tokenization.

Text is split into tokens with a regular expression:
    - a number, kept whole when it has a decimal point or a thousands comma ("3.5", "1,200"),
    - a word (letters, digits and underscore),
    - any other single character (punctuation, symbols).

Every token keeps its position in the original text. This lets the model's output
be mapped back to the exact characters of the question (see `format.recover_value`).
"""

import re
import unicodedata
from dataclasses import dataclass

TOKEN_RE = re.compile(r"\d+(?:[.,]\d+)*|\w+|[^\w\s]")




Writing /content/code/src/t2s/data/text.py


**`Token`** holds the normalized text (what the model uses) and the start and end positions in the original text.

In [4]:
%%writefile -a {CODE}/src/t2s/data/text.py
@dataclass(frozen=True)
class Token:
    text: str  # normalized form, used by the model
    start: int  # position in the original text
    end: int




Appending to /content/code/src/t2s/data/text.py


**`normalize`** does two things. The Unicode NFKC form makes look-alike characters equal (for example the ligature `ﬁ`
becomes `fi`). Then the text is lowercased. Lowercasing is safe for this task: the database text is lowercase, and the
official evaluation lowercases values before comparing them.

In [5]:
%%writefile -a {CODE}/src/t2s/data/text.py
def normalize(text: str) -> str:
    """Unicode NFKC form, then lowercase.

    NFKC makes look-alike characters equal (for example the ligature 'ﬁ' and 'fi').
    Lowercasing is safe for this task: the database text is lowercase and the
    official evaluation lowercases values before comparing.
    """
    return unicodedata.normalize("NFKC", text).lower()




Appending to /content/code/src/t2s/data/text.py


**`tokenize`** applies the pattern and builds the tokens, with their positions.

In [6]:
%%writefile -a {CODE}/src/t2s/data/text.py
def tokenize(text: str) -> list[Token]:
    return [Token(normalize(m.group()), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]




Appending to /content/code/src/t2s/data/text.py


**`words`** gives only the normalized token strings.

In [7]:
%%writefile -a {CODE}/src/t2s/data/text.py
def words(text: str) -> list[str]:
    """Only the normalized token strings."""
    return [t.text for t in tokenize(text)]




Appending to /content/code/src/t2s/data/text.py


**`find_span`** looks for a list of tokens (for example the words of a value) inside another list of tokens (for example the
question), as consecutive tokens. It returns the start and end positions, or `None`.

In [8]:
%%writefile -a {CODE}/src/t2s/data/text.py
def find_span(tokens: list[str], pattern: list[str]) -> tuple[int, int] | None:
    """First position where `pattern` appears as a run of consecutive tokens.

    Returns (start, end) as token indexes, end excluded, or None.
    """
    n = len(pattern)
    if n == 0:
        return None
    for i in range(len(tokens) - n + 1):
        if tokens[i : i + n] == pattern:
            return i, i + n
    return None


Appending to /content/code/src/t2s/data/text.py


In [9]:
from t2s.data.text import tokenize

text = "What position does the player who played for Butler CC (KS) play?"
for t in tokenize(text):
    print(f"{t.text!r:12} characters {t.start:2d}-{t.end:2d}   original text: {text[t.start:t.end]!r}")

print()
print([t.text for t in tokenize("scored 3.5 goals, 1,200 fans")])

'what'       characters  0- 4   original text: 'What'
'position'   characters  5-13   original text: 'position'
'does'       characters 14-18   original text: 'does'
'the'        characters 19-22   original text: 'the'
'player'     characters 23-29   original text: 'player'
'who'        characters 30-33   original text: 'who'
'played'     characters 34-40   original text: 'played'
'for'        characters 41-44   original text: 'for'
'butler'     characters 45-51   original text: 'Butler'
'cc'         characters 52-54   original text: 'CC'
'('          characters 55-56   original text: '('
'ks'         characters 56-58   original text: 'KS'
')'          characters 58-59   original text: ')'
'play'       characters 60-64   original text: 'play'
'?'          characters 64-65   original text: '?'

['scored', '3.5', 'goals', ',', '1,200', 'fans']


**What I observe.** Words are lowercased, every parenthesis is its own token, the numbers `3.5` and `1,200` stay whole, and
each token points back to its characters in the original text (`Butler` becomes `butler` but the original `Butler` is still
reachable through positions 45-51).

## 2. The format of the input and the output

The model needs one input sequence and one output sequence. I define them here, together with the way back from the
output sequence to a query.

- **Input:** `question words <sep> <col> first column <col> second column ...`. The column names are in the input
  because the output has to name them.
- **Output:** `<sel> <agg_count> column <where> column <eq> value words <and> column <gt> value words`.

The tokens in angle brackets are **special tokens**. They are the only separators, so the output can always be cut into
its parts without ambiguity.

**The description of the format** is in the docstring. The imports use the tokenizer above and the `Example` and `Query` of
notebook 01.

In [10]:
%%writefile {CODE}/src/t2s/data/format.py
"""The token format of the model: how a question + table become the input sequence,
and how a query becomes the output sequence (and back).

Input (source):
    question words <sep> <col> first column name <col> second column name ...

Output (target):
    <sel> <agg_count> selected column <where> column <eq> value words <and> column <gt> value words

Special tokens (in angle brackets) are the only separators. Column names and values are
ordinary words, so the output can be split into parts without ambiguity. The agg token is
always present (`<agg_none>` when there is no aggregator); `<where>` is missing when the
query has no condition.

Condition values are written as words, but the database needs the exact characters.
`recover_value` finds the generated words in the question and returns the original text
between the first and last word, so punctuation and spacing are exactly as in the question.
"""

import re

from t2s.data.text import find_span, tokenize, words
from t2s.data.wikisql import Example
from t2s.query import AGG_OPS, Condition, Query



Writing /content/code/src/t2s/data/format.py


**The special tokens.**

- `<pad>` fills short sequences up to the length of the batch; `<unk>` stands for an unknown word; `<sos>` and `<eos>` mark
  the start and the end of the output.
- `<sep>` and `<col>` structure the input; `<sel>`, `<where>`, `<and>`, one `<agg_...>` token per aggregator and one
  operator token (`<eq>`, `<gt>`, `<lt>`) structure the output.
- `SPECIAL_TOKENS` is the full list, and `_STRUCTURE` is the set of structure tokens (used to reject an output where a
  structure token appears inside a column name or a value).

In [11]:
%%writefile -a {CODE}/src/t2s/data/format.py
# Padding, unknown word, start and end of the output sequence.
PAD, UNK, SOS, EOS = "<pad>", "<unk>", "<sos>", "<eos>"

# Structure of the input and output.
SEP, COL = "<sep>", "<col>"
SEL, WHERE, AND = "<sel>", "<where>", "<and>"
AGG_TOKENS = tuple(f"<agg_{op.lower() or 'none'}>" for op in AGG_OPS)
OP_TOKENS = ("<eq>", "<gt>", "<lt>")  # '=', '>' and '<'; WikiSQL's fourth operator never occurs

SPECIAL_TOKENS = (PAD, UNK, SOS, EOS, SEP, COL, SEL, WHERE, AND, *AGG_TOKENS, *OP_TOKENS)

_STRUCTURE = {SEL, WHERE, AND, SEP, COL, *AGG_TOKENS, *OP_TOKENS}




Appending to /content/code/src/t2s/data/format.py


**`source_tokens`** builds the input: the words of the question, then `<sep>`, then each column name introduced by `<col>`.

In [12]:
%%writefile -a {CODE}/src/t2s/data/format.py
def source_tokens(question: str, header: tuple[str, ...] | list[str]) -> list[str]:
    tokens = words(question) + [SEP]
    for name in header:
        tokens += [COL] + words(name)
    return tokens




Appending to /content/code/src/t2s/data/format.py


**`target_tokens`** builds the output of a query: `<sel>`, the aggregator token, and the words of the selected column; then,
for each condition, `<where>` (for the first) or `<and>` (for the next ones), the words of the column, the operator token, and
the words of the value.

In [13]:
%%writefile -a {CODE}/src/t2s/data/format.py
def target_tokens(query: Query, header: tuple[str, ...] | list[str]) -> list[str]:
    tokens = [SEL, AGG_TOKENS[query.agg], *words(header[query.select])]
    for i, cond in enumerate(query.conds):
        if cond.op >= len(OP_TOKENS):
            raise ValueError(f"unsupported operator index {cond.op}")
        tokens.append(WHERE if i == 0 else AND)
        tokens += [*words(header[cond.column]), OP_TOKENS[cond.op], *words(str(cond.value))]
    return tokens




Appending to /content/code/src/t2s/data/format.py


**`recover_value`** solves a practical problem. The model writes a value as words (`butler cc ( ks )`), but the database needs
the exact characters (`butler cc (ks)`): one different space gives a different query. Every value is a piece of the question,
so I look for the written words in the question (with `find_span`) and return the **original text** between the first and the
last word. If the words are not found, they are joined with spaces.

In [14]:
%%writefile -a {CODE}/src/t2s/data/format.py
def recover_value(value_words: list[str], question: str) -> str:
    """The value as written in the question; falls back to joining the words with spaces."""
    question_tokens = tokenize(question)
    span = find_span([t.text for t in question_tokens], value_words)
    if span is None:
        return " ".join(value_words)
    first, last = question_tokens[span[0]], question_tokens[span[1] - 1]
    return re.sub(r"\s+", " ", question[first.start : last.end]).lower()




Appending to /content/code/src/t2s/data/format.py


**`_column_index`** makes a dictionary from the words of each column name to the index of the column. It is used to recognize
column names in an output sequence.

In [15]:
%%writefile -a {CODE}/src/t2s/data/format.py
def _column_index(header: tuple[str, ...] | list[str]) -> dict[tuple[str, ...], int]:
    index: dict[tuple[str, ...], int] = {}
    for i, name in enumerate(header):
        index.setdefault(tuple(words(name)), i)  # on a repeated name, the first column wins
    return index




Appending to /content/code/src/t2s/data/format.py


**`parse_target`** turns an output sequence back into a `Query`.

1. Cut at `<eos>`; check that the sequence starts with `<sel>` and an aggregator token.
2. Cut the rest at `<where>` and `<and>`: the first part is the selected column, the other parts are the conditions.
3. Find the column by its words. In each condition, find the operator token: what is before it is the column, what is after it
   is the value (given back with `recover_value`).

If anything is wrong (the format is broken, a column does not exist, a value is missing, a structure token is inside a name)
it returns `None`. A `None` counts as an invalid prediction.

In [16]:
%%writefile -a {CODE}/src/t2s/data/format.py
def parse_target(
    tokens: list[str], header: tuple[str, ...] | list[str], question: str
) -> Query | None:
    """Turn an output sequence back into a query. Returns None if it is malformed.

    A malformed sequence is one that breaks the format, or names a column that is not in
    the table.
    """
    if EOS in tokens:
        tokens = tokens[: tokens.index(EOS)]
    if len(tokens) < 3 or tokens[0] != SEL or tokens[1] not in AGG_TOKENS:
        return None
    agg = AGG_TOKENS.index(tokens[1])
    columns = _column_index(header)

    # Split the rest into the selected column and the condition parts.
    body = tokens[2:]
    parts, current = [], []
    for tok in body:
        if tok in (WHERE, AND):
            parts.append(current)
            current = []
        else:
            current.append(tok)
    parts.append(current)

    select = columns.get(tuple(parts[0]))
    if select is None or any(t in _STRUCTURE for t in parts[0]):
        return None

    conds = []
    for part in parts[1:]:
        ops = [i for i, t in enumerate(part) if t in OP_TOKENS]
        if len(ops) != 1:
            return None
        op_pos = ops[0]
        column = columns.get(tuple(part[:op_pos]))
        value_words = part[op_pos + 1 :]
        if column is None or not value_words or any(t in _STRUCTURE for t in value_words):
            return None
        conds.append(Condition(column, OP_TOKENS.index(part[op_pos]), recover_value(value_words, question)))
    return Query(select=select, agg=agg, conds=tuple(conds))




Appending to /content/code/src/t2s/data/format.py


**`roundtrip_failures`** tests the format: it converts each gold query to tokens and back, and returns the examples whose query
changed. If the format loses information, they show up here.

In [17]:
%%writefile -a {CODE}/src/t2s/data/format.py
def roundtrip_failures(examples: list[Example]) -> list[Example]:
    """Examples whose gold query is not recovered after query -> tokens -> query.

    The share of recovered queries is the best a model could reach with this token format.
    """
    failures = []
    for ex in examples:
        tokens = target_tokens(ex.query, ex.header)
        parsed = parse_target(tokens, ex.header, ex.question)
        if parsed is None or parsed.canonical() != ex.query.canonical():
            failures.append(ex)
    return failures


Appending to /content/code/src/t2s/data/format.py


In [18]:
from t2s.data.format import source_tokens, target_tokens

for ex in splits["train"][:3] + [ex for ex in splits["train"] if len(ex.query.conds) == 2][:1]:
    print("Question:", ex.question.strip())
    print("SQL     :", ex.query.to_sql(ex.header))
    print("INPUT   :", " ".join(source_tokens(ex.question, ex.header)))
    print("OUTPUT  :", " ".join(target_tokens(ex.query, ex.header)))
    print()

Question: Tell me what the notes are for South Australia
SQL     : SELECT Notes FROM table WHERE Current slogan = 'SOUTH AUSTRALIA'
INPUT   : tell me what the notes are for south australia <sep> <col> state / territory <col> text / background colour <col> format <col> current slogan <col> current series <col> notes
OUTPUT  : <sel> <agg_none> notes <where> current slogan <eq> south australia

Question: What is the current series where the new series began in June 2011?
SQL     : SELECT Current series FROM table WHERE Notes = 'New series began in June 2011'
INPUT   : what is the current series where the new series began in june 2011 ? <sep> <col> state / territory <col> text / background colour <col> format <col> current slogan <col> current series <col> notes
OUTPUT  : <sel> <agg_none> current series <where> notes <eq> new series began in june 2011

Question: What is the format for South Australia?
SQL     : SELECT Format FROM table WHERE State/territory = 'South Australia'
INPUT   : wh

**What I observe.** The input holds the question and all the column names. The output names the columns with ordinary words and
uses special tokens only for the structure. A query with two conditions has `<where>` once and `<and>` once.

### The way back: from tokens to a query

In [19]:
from t2s.data.format import parse_target

ex = next(e for e in splits["dev"] if "(" in e.question and e.query.conds)
tokens = target_tokens(ex.query, ex.header)
print("Question     :", ex.question)
print("Output tokens:", " ".join(tokens))
print("Parsed query :", parse_target(tokens, ex.header, ex.question).to_sql(ex.header))
print("Gold query   :", ex.query.to_sql(ex.header))

Question     : What position does the player who played for butler cc (ks) play?
Output tokens: <sel> <agg_none> position <where> school / club team <eq> butler cc ( ks )
Parsed query : SELECT Position FROM table WHERE School/Club Team = 'butler cc (ks)'
Gold query   : SELECT Position FROM table WHERE School/Club Team = 'Butler CC (KS)'


**What I observe.** The tokens contain `butler cc ( ks )` with spaces around the parentheses, but the parsed query has
`'butler cc (ks)'`, exactly as in the question. It differs from the gold value `Butler CC (KS)` only by upper/lower case, which
the database ignores.

### Does the format lose anything?

I convert every gold query to tokens and back and count how many come back unchanged (the order of the conditions is
ignored). This is the best score any model could reach with this format. For dev I also run the converted query on the database
and compare its answer with the gold answer.

In [20]:
from t2s.data.format import roundtrip_failures

stats["roundtrip"] = {}
for name, examples in splits.items():
    bad = roundtrip_failures(examples)
    stats["roundtrip"][name] = {"queries": len(examples), "recovered": len(examples) - len(bad)}
    print(f"{name:6s} recovered {len(examples) - len(bad):6d} of {len(examples)} ({1 - len(bad) / len(examples):.3%})")
    for ex in bad[:2]:
        print("   not recovered:", ex.question, "|", [c.value for c in ex.query.conds])

train  recovered  56351 of 56355 (99.993%)
   not recovered: When saif saaeed shaheen (qat) is the saif saaeed shaheen ( qat ) what is the world record? | ['Saif Saaeed Shaheen ( QAT )']
   not recovered: What is the 12::37.35 when hengelo , netherlands is hengelo, netherlands? | ['Hengelo, Netherlands']
dev    recovered   8421 of 8421 (100.000%)
test   recovered  15877 of 15878 (99.994%)
   not recovered: When has Result of 2–0, and a Score of 2 – 0? | ['2–0', '2 – 0']


In [21]:
from t2s.data.wikisql import db_path
from t2s.eval.executor import SqliteExecutor

executor = SqliteExecutor(db_path(data_dir, "dev"))
same = 0
for ex in splits["dev"]:
    parsed = parse_target(target_tokens(ex.query, ex.header), ex.header, ex.question)
    same += executor.execute(ex.table_id, parsed) == executor.execute(ex.table_id, ex.query)

stats["execution_oracle_dev"] = {"queries": len(splits["dev"]), "same_answer": same}
print(f"dev: the converted query gives the same answer as the gold query for {same} of {len(splits['dev'])}")

dev: the converted query gives the same answer as the gold query for 8421 of 8421


**What I observe.** The query is recovered for 99.993% of train, **100% of dev** and 99.994% of test (5 failures in total, all
annotation oddities: for example the gold value is `Saif Saaeed Shaheen ( QAT )` while the question reads `(qat)`). On dev, the
converted query gives the same answer as the gold query for all 8,421 questions. So the format and the executor can be trusted:
from now on, an error comes from the model and not from the format.

## 3. The vocabulary

The vocabulary gives an integer id to each token. Two rules:

- it is built from the **training set only**, so the words of dev and test that the model has never seen stay unknown;
- words that are too rare are left out and become `<unk>`.

One vocabulary is shared by the input and the output, because the words written in the output (column names and values) are
words of the input.

**The description and the imports.**

In [22]:
%%writefile {CODE}/src/t2s/data/vocab.py
"""Vocabulary: the mapping between tokens and integer ids.

One vocabulary is shared by the input and the output, because column names and values
written in the output are words that also appear in the input.
"""

import json
from collections import Counter
from pathlib import Path
from typing import Iterable

from t2s.data.format import EOS, PAD, SOS, SPECIAL_TOKENS, UNK




Writing /content/code/src/t2s/data/vocab.py


**`Vocab`** keeps the list of tokens (the position of a token in the list is its id) and a dictionary from token to id. It checks
that there is no duplicate and that every special token is present.

In [23]:
%%writefile -a {CODE}/src/t2s/data/vocab.py
class Vocab:
    def __init__(self, tokens: list[str]):
        self.itos = list(tokens)
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        if len(self.stoi) != len(self.itos):
            raise ValueError("duplicate tokens in vocabulary")
        for special in SPECIAL_TOKENS:
            if special not in self.stoi:
                raise ValueError(f"missing special token {special}")



Appending to /content/code/src/t2s/data/vocab.py


**`build`** makes a vocabulary from sequences of tokens: first the special tokens (so they always have the same ids), then every
word seen at least `min_freq` times. The words are ordered by frequency, and by alphabet when the frequency is equal, so the
result is always exactly the same.

In [24]:
%%writefile -a {CODE}/src/t2s/data/vocab.py
    @classmethod
    def build(cls, sequences: Iterable[list[str]], min_freq: int = 2) -> "Vocab":
        """Special tokens first, then every word seen at least `min_freq` times.

        Words are ordered by frequency (ties alphabetically) so the result is deterministic.
        """
        counts = Counter(tok for seq in sequences for tok in seq)
        specials = set(SPECIAL_TOKENS)
        words = [w for w, c in counts.items() if c >= min_freq and w not in specials]
        words.sort(key=lambda w: (-counts[w], w))
        return cls(list(SPECIAL_TOKENS) + words)



Appending to /content/code/src/t2s/data/vocab.py


The **size** of the vocabulary and the **ids** of the four main special tokens.

In [25]:
%%writefile -a {CODE}/src/t2s/data/vocab.py
    def __len__(self) -> int:
        return len(self.itos)

    @property
    def pad_id(self) -> int:
        return self.stoi[PAD]

    @property
    def unk_id(self) -> int:
        return self.stoi[UNK]

    @property
    def sos_id(self) -> int:
        return self.stoi[SOS]

    @property
    def eos_id(self) -> int:
        return self.stoi[EOS]



Appending to /content/code/src/t2s/data/vocab.py


**`encode`** turns tokens into ids. A word that is not in the vocabulary gets the id of `<unk>`.

In [26]:
%%writefile -a {CODE}/src/t2s/data/vocab.py
    def encode(self, tokens: list[str]) -> list[int]:
        """Unknown words become the id of <unk>."""
        unk = self.unk_id
        return [self.stoi.get(t, unk) for t in tokens]



Appending to /content/code/src/t2s/data/vocab.py


**`decode`** turns ids back into tokens.

In [27]:
%%writefile -a {CODE}/src/t2s/data/vocab.py
    def decode(self, ids: Iterable[int]) -> list[str]:
        return [self.itos[i] for i in ids]



Appending to /content/code/src/t2s/data/vocab.py


**`save` and `load`** write and read the vocabulary as a JSON file. (Every trained model also stores its own copy of the
vocabulary, see notebook 03.)

In [28]:
%%writefile -a {CODE}/src/t2s/data/vocab.py
    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(self.itos, ensure_ascii=False))

    @classmethod
    def load(cls, path: str | Path) -> "Vocab":
        return cls(json.loads(Path(path).read_text()))




Appending to /content/code/src/t2s/data/vocab.py


**`unknown_share`** measures how much of some data the vocabulary cannot represent: the share of unknown tokens, and the share of
sequences that contain at least one unknown token.

In [29]:
%%writefile -a {CODE}/src/t2s/data/vocab.py
def unknown_share(vocab: Vocab, sequences: Iterable[list[str]]) -> dict[str, float]:
    """How much of the data the vocabulary cannot represent.

    'tokens': share of tokens that are unknown.
    'sequences': share of sequences with at least one unknown token.
    """
    total = unknown = with_unknown = n = 0
    for seq in sequences:
        n += 1
        misses = sum(1 for t in seq if t not in vocab.stoi)
        total += len(seq)
        unknown += misses
        with_unknown += misses > 0
    return {"tokens": unknown / total, "sequences": with_unknown / n}


Appending to /content/code/src/t2s/data/vocab.py


### The minimum frequency

A larger vocabulary means a larger embedding table and output layer, a smaller one means more unknown words. The table shows the
trade-off for several values of `min_freq`. Frequencies are counted on the **input side only**: a rare word appears once in the
input and once in the output, so counting both sides would make every rare word look twice as frequent.

In [30]:
import pandas as pd
from t2s.data.vocab import Vocab, unknown_share

train_inputs = [source_tokens(e.question, e.header) for e in splits["train"]]
dev_inputs = [source_tokens(e.question, e.header) for e in splits["dev"]]
dev_outputs = [target_tokens(e.query, e.header) for e in splits["dev"]]
test_outputs = [target_tokens(e.query, e.header) for e in splits["test"]]

rows = []
for min_freq in [1, 2, 3, 5, 10]:
    v = Vocab.build(train_inputs, min_freq=min_freq)
    rows.append({
        "min_freq": min_freq,
        "vocabulary size": len(v),
        "dev input: unknown tokens": f"{unknown_share(v, dev_inputs)['tokens']:.1%}",
        "dev output: unknown tokens": f"{unknown_share(v, dev_outputs)['tokens']:.1%}",
        "dev queries with an unknown word": f"{unknown_share(v, dev_outputs)['sequences']:.1%}",
        "test queries with an unknown word": f"{unknown_share(v, test_outputs)['sequences']:.1%}",
    })
stats["vocabulary_by_min_freq"] = rows
pd.DataFrame(rows)

,min_freq,vocabulary size,dev input: unknown tokens,dev output: unknown tokens,dev queries with an unknown word,test queries with an unknown word
0,1,35568,1.9%,3.7%,31.2%,32.1%
1,2,15962,2.4%,4.7%,38.0%,38.8%
2,3,11636,2.7%,5.3%,41.7%,43.0%
3,5,8560,3.2%,6.2%,46.6%,48.1%
4,10,5259,4.0%,7.5%,53.0%,55.1%


**What I observe.** Look at `min_freq = 2`: only 2.4% of the dev input tokens
and 4.7% of the dev output tokens are unknown, which looks small. But **38.0% of the dev queries contain at least one unknown
word**. A query has about 13 output tokens, so a 4.7% chance per token becomes one unknown word in more than a third of the
queries.

The output has more unknown words than the input because the input is full of common words (`the`, `what`, `is`), while the
output holds the rare ones: column names and values.

A model that writes words from this vocabulary can only write `<unk>` for such a word, which is never correct. **So its score on
dev cannot exceed about 62%**, however good the model is. A bigger vocabulary (`min_freq = 1`) only brings it down to 31%. This is the
reason for the copy mechanism I add later. I choose `min_freq = 2`, a usual value.

## 4. Ids, truncation, padding and batches

The last step turns the tokens of an example into ids, and groups examples in batches (matrices) that the network can process
at once.

**The layout of the output sequence.** It is stored as `<sos> ... <eos>`. For training, the decoder input is the sequence without
its last token and the expected output is the sequence without its first token, so at each position the decoder must predict
the next token (this is *teacher forcing*, see the concepts note).

In [31]:
%%writefile {CODE}/src/t2s/data/dataset.py
"""From examples to batches of padded tensors.

The target sequence is stored with <sos> at the start and <eos> at the end. For training,
the decoder input is the target without its last token and the expected output is the target
without its first token, so at every position the model predicts the next token.

    target    : <sos>  <sel>  <agg_none>  player  <eos>
    dec. input: <sos>  <sel>  <agg_none>  player
    expected  : <sel>  <agg_none>  player  <eos>
"""

import random
from dataclasses import dataclass

from t2s.data.format import source_tokens, target_tokens
from t2s.data.vocab import Vocab
from t2s.data.wikisql import Example




Writing /content/code/src/t2s/data/dataset.py


**`Encoded`** is one example as two lists of ids: the input and the output.

In [32]:
%%writefile -a {CODE}/src/t2s/data/dataset.py
@dataclass(frozen=True)
class Encoded:
    src: list[int]
    tgt: list[int]  # <sos> ... <eos>




Appending to /content/code/src/t2s/data/dataset.py


**`build_vocab`** is the function used by all the models to build the vocabulary. It uses the **training set only**, and the
**input side only** (every word of the output is also a word of the input, and a rare word is not counted twice).

In [33]:
%%writefile -a {CODE}/src/t2s/data/dataset.py
def build_vocab(train_examples: list[Example], min_freq: int = 2) -> Vocab:
    """The vocabulary used by all models.

    It is built from the input side of the training set only. Every word of the output
    (column names and values) is also a word of the input, so nothing is lost, and counting
    only the input avoids counting a rare word twice (once in the input, once in the output).
    The dev and test sets are never used, so their words stay unknown.
    """
    return Vocab.build(
        (source_tokens(ex.question, ex.header) for ex in train_examples), min_freq=min_freq
    )




Appending to /content/code/src/t2s/data/dataset.py


**`encode_example`** turns an example into ids. Sequences longer than the limits are cut at the end (the output keeps its
`<sos>` and `<eos>`). The limits are chosen in section 4.2 so that almost nothing is cut.

In [34]:
%%writefile -a {CODE}/src/t2s/data/dataset.py
def encode_example(
    ex: Example, vocab: Vocab, max_src_len: int = 128, max_tgt_len: int = 64
) -> Encoded:
    """Convert one example to ids. Sequences longer than the limits are cut at the end."""
    src = vocab.encode(source_tokens(ex.question, ex.header))[:max_src_len]
    tgt = vocab.encode(target_tokens(ex.query, ex.header))[: max_tgt_len - 2]
    return Encoded(src, [vocab.sos_id, *tgt, vocab.eos_id])




Appending to /content/code/src/t2s/data/dataset.py


**`make_batches`** groups examples of similar length. Short and long sequences in the same batch waste computation on padding.
So the examples are shuffled, cut into pools of 50 batches, sorted by length inside each pool, and split into batches; then the
batches are shuffled.

In [35]:
%%writefile -a {CODE}/src/t2s/data/dataset.py
def make_batches(
    lengths: list[int], batch_size: int, shuffle: bool, seed: int = 0, pool: int = 50
) -> list[list[int]]:
    """Group example indexes into batches of similar length.

    Short and long sequences in one batch waste computation on padding. The indexes are
    shuffled, cut into pools of `pool` batches, sorted by length inside each pool, and then
    split into batches. The batches themselves are shuffled for training.
    """
    rng = random.Random(seed)
    order = list(range(len(lengths)))
    if shuffle:
        rng.shuffle(order)
    size = batch_size * pool
    batches = []
    for i in range(0, len(order), size):
        chunk = sorted(order[i : i + size], key=lambda j: lengths[j])
        batches += [chunk[k : k + batch_size] for k in range(0, len(chunk), batch_size)]
    if shuffle:
        rng.shuffle(batches)
    return batches




Appending to /content/code/src/t2s/data/dataset.py


**`padding_share`** measures the share of padding in the padded batches (wasted computation).

In [36]:
%%writefile -a {CODE}/src/t2s/data/dataset.py
def padding_share(lengths: list[int], batches: list[list[int]]) -> float:
    """Share of positions in the padded batches that are padding (wasted computation)."""
    real = sum(lengths[i] for b in batches for i in b)
    padded = sum(max(lengths[i] for i in b) * len(b) for b in batches)
    return 1 - real / padded




Appending to /content/code/src/t2s/data/dataset.py


**`collate`** pads a list of encoded examples with `<pad>` and builds the tensors of a batch: `src` (the inputs), `src_len` (their real
lengths), `tgt_in` (the decoder input), `tgt_out` (the expected output) and `tgt_len`. PyTorch is imported here, inside the
function, so that the rest of the file can be used without it.

In [37]:
%%writefile -a {CODE}/src/t2s/data/dataset.py
def collate(items: list[Encoded], pad_id: int) -> dict:
    """Pad a list of encoded examples into tensors. Needs PyTorch."""
    import torch

    def pad(seqs: list[list[int]]) -> "torch.Tensor":
        longest = max(len(s) for s in seqs)
        return torch.tensor([s + [pad_id] * (longest - len(s)) for s in seqs], dtype=torch.long)

    src = [it.src for it in items]
    tgt = [it.tgt for it in items]
    return {
        "src": pad(src),  # (B, T)
        "src_len": torch.tensor([len(s) for s in src]),  # (B,)
        "tgt_in": pad([t[:-1] for t in tgt]),  # (B, U)
        "tgt_out": pad([t[1:] for t in tgt]),  # (B, U)
        "tgt_len": torch.tensor([len(t) - 1 for t in tgt]),  # (B,)
    }




Appending to /content/code/src/t2s/data/dataset.py


**`PlainCodec`** bundles everything that turns text into ids and back: the vocabulary, the encoding of an example (`encode`), and the
decoding of the output of a model into tokens (`decode`). Here every word is one token. The code that trains and evaluates the models only
uses a *codec*, so later (notebook 05) the copy mechanism and a BPE vocabulary can be added as two other codecs, without changing the training
code. `state` and `from_state` save and rebuild a codec (a trained model keeps its own codec).

In [38]:
%%writefile -a {CODE}/src/t2s/data/dataset.py
class PlainCodec:
    """The simplest way to turn examples into ids, and the output of a model back into tokens.

    Each word is one token and the vocabulary comes from the training set. A codec bundles
    the vocabulary, the encoding of an example and the decoding of an output. The copy
    mechanism and the BPE vocabulary (notebook 05) are two other codecs with the same methods,
    so the training and evaluation code does not depend on the way text is turned into ids.
    """

    kind = "plain"

    def __init__(self, vocab: Vocab):
        self.vocab = vocab

    @classmethod
    def build(cls, train_examples: list[Example], data_cfg: dict) -> "PlainCodec":
        return cls(build_vocab(train_examples, data_cfg["min_freq"]))

    def encode(self, ex: Example, max_src_len: int = 128, max_tgt_len: int = 64) -> Encoded:
        return encode_example(ex, self.vocab, max_src_len, max_tgt_len)

    def decode(self, ids: list[int], encoded: Encoded | None = None) -> list[str]:
        """Output ids to word tokens. `encoded` is the encoded input (used by other codecs)."""
        return self.vocab.decode(ids)

    def state(self) -> dict:
        """What is saved with a trained model, to rebuild the codec."""
        return {"kind": self.kind, "vocab": self.vocab.itos}

    @classmethod
    def from_state(cls, state: dict) -> "PlainCodec":
        return cls(Vocab(state["vocab"]))


Appending to /content/code/src/t2s/data/dataset.py


### 4.1 The chosen vocabulary

In [39]:
from t2s.data.dataset import build_vocab
from t2s.data.format import SPECIAL_TOKENS

vocab = build_vocab(splits["train"], min_freq=2)
n_special = len(SPECIAL_TOKENS)
print("vocabulary size:", len(vocab), f"({n_special} special tokens + {len(vocab) - n_special} words)")
print("special tokens :", {tok: vocab.stoi[tok] for tok in SPECIAL_TOKENS})
print("most frequent words:", vocab.itos[n_special:n_special + 15])
stats["vocabulary"] = {"min_freq": 2, "size": len(vocab), "special_tokens": n_special}

example = splits["dev"][0]
tokens = source_tokens(example.question, example.header)
print()
print("tokens:", tokens)
print("ids   :", vocab.encode(tokens))

vocabulary size: 15962 (18 special tokens + 15944 words)
special tokens : {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3, '<sep>': 4, '<col>': 5, '<sel>': 6, '<where>': 7, '<and>': 8, '<agg_none>': 9, '<agg_max>': 10, '<agg_min>': 11, '<agg_count>': 12, '<agg_sum>': 13, '<agg_avg>': 14, '<eq>': 15, '<gt>': 16, '<lt>': 17}
most frequent words: ['the', '?', 'what', 'of', 'is', 'team', '(', ')', 'date', 'score', 'a', ',', 'was', 'for', '.']

tokens: ['what', 'position', 'does', 'the', 'player', 'who', 'played', 'for', 'butler', 'cc', '(', 'ks', ')', 'play', '?', '<sep>', '<col>', 'player', '<col>', 'no', '.', '<col>', 'nationality', '<col>', 'position', '<col>', 'years', 'in', 'toronto', '<col>', 'school', '/', 'club', 'team']
ids   : [20, 49, 297, 18, 47, 63, 83, 31, 6407, 639, 24, 9205, 25, 209, 19, 4, 5, 47, 5, 68, 32, 5, 130, 5, 49, 5, 135, 33, 873, 5, 105, 38, 81, 23]


**What I observe.** The vocabulary has 15,962 entries: 18 special tokens and 15,944 words. The ids follow the frequency: `the` has a
small id, a rare word such as `butler` has a large one. An id has **no meaning** by itself; the meaning will come from the
embedding that the network learns for each id.

### 4.2 Lengths, truncation

In [40]:
import numpy as np
from t2s.data.dataset import encode_example, make_batches, padding_share, collate

lengths = {
    "input": [len(t) for t in train_inputs],
    "output": [len(target_tokens(e.query, e.header)) + 2 for e in splits["train"]],  # +2 for <sos> and <eos>
}
for name, values in lengths.items():
    v = np.array(values)
    print(f"{name}: mean {v.mean():.1f}, median {np.median(v):.0f}, 95th percentile {np.percentile(v, 95):.0f}, "
          f"99th percentile {np.percentile(v, 99):.0f}, max {v.max()}")

MAX_SRC, MAX_TGT = 128, 64
cut_src = sum(l > MAX_SRC for l in lengths["input"])
cut_tgt = sum(l > MAX_TGT for l in lengths["output"])
print(f"\nWith limits {MAX_SRC} (input) and {MAX_TGT} (output), the training set loses data in "
      f"{cut_src} inputs and {cut_tgt} outputs of {len(train_inputs)}.")
stats["lengths"] = {"max_src": MAX_SRC, "max_tgt": MAX_TGT, "inputs_cut": cut_src, "outputs_cut": cut_tgt}

input: mean 32.4, median 30, 95th percentile 50, 99th percentile 69, max 190
output: mean 13.6, median 13, 95th percentile 22, 99th percentile 27, max 93

With limits 128 (input) and 64 (output), the training set loses data in 28 inputs and 4 outputs of 56355.


**What I observe.** Inputs have 32 tokens on average (99% have 69 or fewer, the longest has 190) and outputs have 14 on average
(the longest has 93). With the limits 128 and 64, only 28 inputs and 4 outputs out of 56,355 lose their end, so I keep these limits.

### 4.3 Batches and padding

In [41]:
import random

BATCH = 64
src_lengths = [min(len(t), MAX_SRC) for t in train_inputs]

order = list(range(len(src_lengths)))
random.Random(0).shuffle(order)
random_batches = [order[i:i + BATCH] for i in range(0, len(order), BATCH)]
grouped_batches = make_batches(src_lengths, BATCH, shuffle=True, seed=0)

stats["padding_share"] = {
    "random_batches": padding_share(src_lengths, random_batches),
    "length_grouped_batches": padding_share(src_lengths, grouped_batches),
}
print(f"share of padding in the input: random batches {stats['padding_share']['random_batches']:.1%}, "
      f"length-grouped batches {stats['padding_share']['length_grouped_batches']:.1%}")

share of padding in the input: random batches 56.1%, length-grouped batches 4.2%


In [42]:
encoded = [encode_example(e, vocab, MAX_SRC, MAX_TGT) for e in splits["train"][:2000]]
first = make_batches([len(x.src) for x in encoded], 4, shuffle=False)[0]
batch = collate([encoded[i] for i in first], vocab.pad_id)
for key, tensor in batch.items():
    print(f"{key:8s} {tuple(tensor.shape)}")
print()
print("decoder input   :", vocab.decode(batch["tgt_in"][0].tolist()))
print("expected output :", vocab.decode(batch["tgt_out"][0].tolist()))

src      (4, 20)
src_len  (4,)
tgt_in   (4, 9)
tgt_out  (4, 9)
tgt_len  (4,)

decoder input   : ['<sos>', '<sel>', '<agg_none>', 'actors', 'name', '<where>', 'country', '<eq>', 'ukraine']
expected output : ['<sel>', '<agg_none>', 'actors', 'name', '<where>', 'country', '<eq>', 'ukraine', '<eos>']


**What I observe.**

- In random batches **56.1%** of the input positions are padding; in length-grouped batches only **4.2%**. The network does about
  half the useless work.
- A batch is a set of matrices: `src (4, 20)` means 4 inputs padded to 20 tokens; `tgt_in` and `tgt_out` have the same shape.
- The decoder input and the expected output are the same sequence **shifted by one position**: the decoder reads `<sos>`
  and must predict `<sel>`, reads `<sel>` and must predict `<agg_none>`, and so on until `<eos>`.

### 4.4 The codec

All of the above is used through a codec. I check that encoding an example and decoding its output gives back the tokens of the target.

In [43]:
from t2s.data.dataset import PlainCodec

codec = PlainCodec.build(splits["train"], {"min_freq": 2})
ex = splits["dev"][5]
encoded = codec.encode(ex)
print("target ids     :", encoded.tgt)
print("decoded tokens :", " ".join(codec.decode(encoded.tgt[1:-1], encoded)))
print("same as target_tokens:", codec.decode(encoded.tgt[1:-1], encoded) == target_tokens(ex.query, ex.header))

target ids     : [2, 6, 9, 47, 7, 105, 38, 81, 23, 15, 1, 72, 105, 3]
decoded tokens : <sel> <agg_none> player <where> school / club team <eq> <unk> high school
same as target_tokens: False


**What I observe.** Decoding the ids of the target gives back exactly the tokens of the target (when a word is in the vocabulary). With
the plain codec, a word that is not in the vocabulary would be decoded as `<unk>`; this is the limit that notebook 05 addresses.

## 5. Summary of this notebook

**What I built:** a tokenizer with positions, the format of the input and output sequences (with the way back to a query), a
vocabulary, and the conversion of examples to padded batches of ids.

**What I learned:**

1. The format loses almost nothing: 99.99% of the gold queries come back unchanged, and on dev the converted query gives the same
   answer as the gold query for every question.
2. The vocabulary is the main limit. Even though only 4.7% of the output tokens are unknown, **38% of the dev queries contain an
   unknown word**, so a model that writes words from the vocabulary cannot exceed about 62% on dev.
3. Grouping examples by length reduces the padding from 56% to 4%.

**Next notebook:** `03_lstm_baselines.ipynb` builds the first models.

In [44]:
(out_dir / "stats.json").write_text(json.dumps(stats, indent=2))
print("saved", out_dir / "stats.json")

saved results/phase2_text/stats.json
